In [ ]:
import os
import json
import random
import time
import requests
import pandas as pd

API_URL = ""
API_KEY = ""
MODEL = "gpt-5.2"
FILE_NAME = "intent_analysis_train_v2.csv"
BATCH_SIZE = 50
TOTAL_BATCHES = 60
MAX_RETRIES = 3

SYSTEM_PROMPT = """You are an expert data annotator for a fashion AI system. Generate synthetic training data for an Intent Classification model (3 classes: graph_pairing, color_variant, similar_items).

ABSOLUTE STRICT RULES (VIOLATION MEANS FAILURE):
1. USE PRONOUNS, OMIT NOUNS: Real users look at a product image when typing. 70% of your queries MUST use "this", "these", "that", "it" instead of explicitly naming the clothing item. (e.g., instead of "jacket to wear with low rise jeans", write "jacket for these").
2. EXTREME FRAGMENTATION: 50% of the queries must be 2-5 words long. Pure laziness. (e.g., "shoes for this", "same in red", "dupe for it", "this but cheaper").
3. SYNTACTIC OVERLAP: Force the use of "for", "with", "to" across ALL THREE classes to confuse the AI.
4. NO FORMULAIC ENDINGS: DO NOT append repetitive reasons at the end of sentences like "for a summer look", "for a retro vibe", "for the exact same style". End the sentence abruptly.

CLASS DEFINITIONS:
1. 'graph_pairing': Wants items to wear ALONGSIDE the reference item. (e.g., "what goes with this", "top for these", "shoes to match")
2. 'color_variant': Wants the EXACT SAME item in a DIFFERENT COLOR. (e.g., "this in blue", "got it in red?", "black version for this")
3. 'similar_items': Wants a DIFFERENT BUT VISUALLY SIMILAR item (cheaper, alternative). (e.g., "dupe for this", "cheaper alt", "similar to these but longer")

Return ONLY a raw JSON array of objects. Schema: [{"query": "query", "intent": "intent"}]"""

FASHION_TOPICS = [
    "winter coats & puffers", "summer swimwear & bikinis", "formal wedding guest attire", 
    "streetwear & hype sneakers", "vintage 90s denim", "gym & activewear", 
    "office suits & blazers", "floral summer dresses", "luxury handbags & totes", 
    "goth & punk boots", "minimalist capsule wardrobe",
    "boho chic maxi dresses", "athleisure & leggings", "preppy knitwear & cardigans",
    "cyberpunk techwear", "old money aesthetic", "loungewear & silk pajamas",
    "festival & rave outfits", "rainy day waterproof gear", "maternity fashion",
    "chunky loafers & oxfords", "corsets & bustiers", "cargo pants & utility wear",
    "statement jewelry & watches", "sunglasses & beach accessories", "leather biker jackets",
    "cashmere sweaters", "linen beach shirts", "platform heels & stilettos",
    "denim shorts & cutoffs", "fleece & sherpa jackets", "trench coats",
    "graphic tees & band shirts", "midi & maxi skirts", "snowboard & ski apparel",
    "velvet party dresses", "sustainable & eco-friendly fashion", "slip dresses",
    "tracksuits & hoodies"
]

print("start")

for i in range(1, TOTAL_BATCHES + 1):
    topic = random.choice(FASHION_TOPICS)
    current_temp = round(random.uniform(0.75, 0.95), 2)
    prompt_modifier = f"Generate {BATCH_SIZE} unique queries focusing on the category: '{topic}'. Batch ID: {i}."
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_modifier}
        ],
        "temperature": current_temp,
        "max_tokens": 2500
    }
    
    success = False
    retries = 0
    
    while not success and retries < MAX_RETRIES:
        try:
            response = requests.post(
                API_URL, 
                headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, 
                json=payload, 
                timeout=60
            )
            
            if response.status_code == 200:
                raw_text = response.json()['choices'][0]['message']['content']
                clean_json = raw_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
                
                data = json.loads(clean_json)
                df = pd.DataFrame(data)
                
                write_header = not os.path.exists(FILE_NAME)
                df.to_csv(FILE_NAME, mode='a', index=False, header=write_header, encoding='utf-8')
                print(f"batch {i}/{TOTAL_BATCHES} saved")
                success = True
            else:
                retries += 1
                print(f"batch {i} api error: {response.status_code}. Retry {retries}/{MAX_RETRIES}")
                time.sleep(2)
                
        except Exception as e:
            retries += 1
            print(f"batch {i} failed: {e}. Retry {retries}/{MAX_RETRIES}")
            time.sleep(2)
            
    if not success:
        print(f"batch {i} completely failed after {MAX_RETRIES} retries. Moving to next.")

print("done")

start
batch 1/60 saved
batch 2/60 saved
batch 3/60 saved
batch 4/60 saved
batch 5/60 saved
batch 6/60 saved
batch 7/60 saved
batch 8/60 saved
batch 9/60 saved
batch 10/60 saved
batch 11/60 saved
batch 12/60 saved
batch 13/60 saved
batch 14/60 saved
batch 15/60 saved
batch 16/60 saved
batch 17/60 saved
batch 18/60 saved
batch 19/60 saved
batch 20/60 saved
batch 21/60 saved
batch 22/60 saved
batch 23/60 saved
batch 24/60 saved
batch 25/60 saved
batch 26/60 saved
batch 27/60 saved
batch 28/60 saved
batch 29/60 saved
batch 30/60 saved
batch 31/60 saved
batch 32/60 saved
batch 33/60 saved
batch 34/60 saved
batch 35/60 saved
batch 36/60 saved
batch 37/60 saved
batch 38/60 saved
batch 39/60 saved
batch 40/60 saved
batch 41/60 saved
batch 42/60 saved
batch 43/60 saved
batch 44/60 saved
batch 45/60 saved
batch 46/60 saved
batch 47/60 saved
batch 48/60 saved
batch 49/60 saved
batch 50/60 saved
batch 51/60 saved
batch 52/60 saved
batch 53/60 saved
batch 54/60 saved
batch 55/60 saved
batch 56/60 s

In [ ]:
import os
import json
import time
import requests
import pandas as pd

API_URL = ""
API_KEY = ""
MODEL = "gpt-5.4"
INPUT_FILE = "intent_analysis_train_v2.csv"
OUTPUT_FILE = "intent_analysis_train_v3.csv"
BATCH_SIZE = 50
MAX_RETRIES = 3

SYSTEM_PROMPT = """You are a strict data evaluator for a fashion Intent Classification system.
Classify each query into exactly one of these 3 classes: 'graph_pairing', 'color_variant', 'similar_items'.
Return ONLY a raw JSON array.
Schema: [{"index": integer, "predicted_intent": "string"}]"""

df = pd.read_csv(INPUT_FILE)
df['predicted_intent'] = None
df['Is_Ok'] = None

print("evaluation started")

for i in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[i:i+BATCH_SIZE]
    queries_payload = [{"index": int(idx), "query": row['query']} for idx, row in batch.iterrows()]
    
    user_prompt = f"Evaluate these queries:\n{json.dumps(queries_payload, ensure_ascii=False)}"
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.0,
        "max_tokens": 2500
    }
    
    success = False
    retries = 0
    
    while not success and retries < MAX_RETRIES:
        try:
            response = requests.post(API_URL, headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, json=payload, timeout=60)
            if response.status_code == 200:
                raw_text = response.json()['choices'][0]['message']['content']
                clean_json = raw_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
                results = json.loads(clean_json)
                
                for res in results:
                    idx = res['index']
                    pred = res['predicted_intent']
                    df.at[idx, 'predicted_intent'] = pred
                    df.at[idx, 'Is_Ok'] = 1 if pred == df.at[idx, 'intent'] else 0
                    
                success = True
                print(f"batch {i//BATCH_SIZE + 1} processed")
            else:
                retries += 1
                time.sleep(2)
        except Exception:
            retries += 1
            time.sleep(2)

df.to_csv(OUTPUT_FILE, index=False)
print("evaluation completed\n")

total = len(df)
evaluated = df['Is_Ok'].notna().sum()
correct = (df['Is_Ok'] == 1).sum()
incorrect = (df['Is_Ok'] == 0).sum()

print("STATISTICS:")
print(f"Total Rows: {total}")
print(f"Evaluated Rows: {evaluated}")
print(f"Is_Ok = 1 (Matched): {correct}")
print(f"Is_Ok = 0 (Mismatched): {incorrect}")

if evaluated > 0:
    agreement_rate = (correct / evaluated) * 100
    print(f"Agreement Rate: {agreement_rate:.2f}%")

evaluation started
batch 1 processed
batch 2 processed
batch 3 processed
batch 4 processed
batch 5 processed
batch 6 processed
batch 7 processed
batch 8 processed
batch 9 processed
batch 10 processed
batch 11 processed
batch 12 processed
batch 13 processed
batch 14 processed
batch 15 processed
batch 16 processed
batch 17 processed
batch 18 processed
batch 19 processed
batch 20 processed
batch 21 processed
batch 22 processed
batch 23 processed
batch 24 processed
batch 25 processed
batch 26 processed
batch 27 processed
batch 28 processed
batch 29 processed
batch 30 processed
batch 31 processed
batch 32 processed
batch 33 processed
batch 34 processed
batch 35 processed
batch 36 processed
batch 37 processed
batch 38 processed
batch 39 processed
batch 40 processed
batch 41 processed
batch 42 processed
batch 43 processed
batch 44 processed
batch 45 processed
batch 46 processed
batch 47 processed
batch 48 processed
batch 49 processed
batch 50 processed
batch 51 processed
batch 52 processed
ba

Kết quả khá ổn, xóa đi mấy cột mismatch là data sạch

In [4]:

df = pd.read_csv('intent_analysis_train_v3.csv')

df_final = df[df['Is_Ok'] == 1].copy()

df_final = df_final[['query', 'intent']]

df_final.to_csv('intent_analysis_train.csv', index=False)

print(f"final rows: {len(df_final)}")

final rows: 3014


Giờ tạo data cho test -> dùng api khác + dùng 1 vài request để check tránh train với test trùng nhau

Tạo tầm 15 x 15 = 225 là ok

In [ ]:
import os
import json
import random
import requests
import pandas as pd

API_URL = ""
API_KEY = ""
MODEL = "gpt-5.4-mini"
FILE_NAME = "intent_analysis_test.csv"
BATCH_SIZE = 15
TOTAL_BATCHES = 15

TOPICS = [
    "winter coats beanies scarves", "summer beachwear bikinis sunglasses",
    "gym wear leggings sports bras", "formal wedding guest dresses suits",
    "streetwear oversized hoodies sneakers", "office wear blazers trousers",
    "vintage clothes retro jackets", "pajamas loungewear slippers",
    "denim jackets jeans shorts", "party outfits sequin tops heels",
    "raincoats waterproof boots", "jewelry necklaces rings",
    "skatewear baggy pants graphic tees", "boho style maxi dresses sandals",
    "goth punk style leather jackets boots"
]

SYSTEM_PROMPT = """You are an expert data annotator for a fashion AI system. Generate synthetic testing data for an Intent Classification model (3 classes: graph_pairing, color_variant, similar_items).

RULES FOR CLEAR, UNAMBIGUOUS QUERIES:
- Emulate fast mobile typing (e.g., lowercase, missing punctuation, "rn", "pls", "tho", "fit"). Do NOT use exaggerated slang.
- The intent of the user MUST be immediately clear and unambiguous to a human reader. Avoid vague phrases like "matching beanie" (which could mean same color or complementary style).
- Return ONLY a raw JSON array of objects, NO markdown formatting.

STRICT CLASS DEFINITIONS:
1. 'graph_pairing': The user wants a DIFFERENT TYPE of clothing to wear WITH a specific item.
   - Example: "what top goes with these jeans", "need a bag for this dress"
   - Must contain pairing verbs: "go with", "wear with", "for this".

2. 'color_variant': The user wants the EXACT SAME item, but in a SPECIFIC DIFFERENT COLOR.
   - Example: "got this in black?", "red version of this hoodie pls"
   - Must contain a specific color name (red, blue, black, etc.) AND imply it's the same item.

3. 'similar_items': The user wants a DIFFERENT BUT VISUALLY SIMILAR item (cheaper, different brand, slightly different style).
   - Example: "dupe for these boots", "similar dress but longer", "cheaper alt for this jacket"
   - Must contain similarity words: "dupe", "similar", "like this", "alternative".

JSON Schema: [{"query": "short clear query here", "intent": "intent_name"}]"""

In [14]:
def generate_test_batch(topic, temp):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Generate {BATCH_SIZE} diverse fashion queries for topic: '{topic}'."}
        ],
        "temperature": temp,
        "max_tokens": 1000
    }
    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=30)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        return None
    except Exception:
        return None

In [15]:
for i in range(1, TOTAL_BATCHES + 1):
    temp = round(random.uniform(0.75, 0.95), 2)
    topic = TOPICS[i - 1]
    raw_text = generate_test_batch(topic, temp)
    
    if not raw_text:
        print(f"batch {i} failed")
        continue
        
    try:
        clean_json = raw_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        df = pd.DataFrame(json.loads(clean_json))
        
        write_header = not os.path.exists(FILE_NAME)
        df.to_csv(FILE_NAME, mode='a', index=False, header=write_header, encoding='utf-8')
        print(f"batch {i} done")
        
    except json.JSONDecodeError:
        print(f"batch {i} error")

batch 1 done
batch 2 done
batch 3 done
batch 4 done
batch 5 done
batch 6 done
batch 7 done
batch 8 done
batch 9 done
batch 10 done
batch 11 done
batch 12 done
batch 13 done
batch 14 done
batch 15 done


Xóa dup

In [5]:
df = df.drop_duplicates(subset=['query'])
df.to_csv('intent_analysis_test.csv', index=False)
print('done')

done
